In [ ]:
"""
A single vercise electrode
is generated.
It is meshed and the contacts
are colored to highlight their location.
"""

import asyncio
import base64
import os
from dataclasses import asdict

import netgen.occ as occ
from ngsolve import Mesh, TaskManager
from ngsolve.webgui import Draw

import ossdbs

In [ ]:
from netgen.meshing import meshsize

In [ ]:
custom_params = ossdbs.electrodes.default_electrode_parameters[
    "BostonScientificVercise"
]
custom_params.total_length = 40.0
settings = {
    "Electrodes": [
        {
            "Name": "BostonScientificVerciseCustom",
            "CustomParameters": asdict(custom_params),
            "Rotation[Degrees]": 0,
            "Direction": {"x[mm]": 0, "y[mm]": 0, "z[mm]": 1},
            "TipPosition": {"x[mm]": 0, "y[mm]": 0, "z[mm]": 0},
            "TotalLength": 40,
        },
    ],
    "ExportElectrode": False,
    "Mesh": {},
}

electrodes = ossdbs.generate_electrodes(settings)
vercise = electrodes[0]
occgeo = occ.OCCGeometry(vercise.geometry)
with TaskManager():
    mesh = Mesh(occgeo.GenerateMesh(meshsize.coarse))

draw_settings = {
    "Objects": {"Wireframe": True},
    "Misc": {"line_thickness": 24},
}

js_code = """
// Color wireframe (mesh edges) white
const wireframeObj = scene.render_objects.find(o => o.name === "Wireframe");
if (wireframeObj && wireframeObj.three_object && wireframeObj.three_object.material) {
    const mat = wireframeObj.three_object.material;
    mat.fragmentShader = mat.fragmentShader.replace(
        'gl_FragColor = vec4(0,0,0, 1);',
        'gl_FragColor = vec4(1,1,1, 1);'
    );
    mat.needsUpdate = true;
}

// Black background: override the method called on every render frame
scene.getBackgroundColor = () => new modules.THREE.Color(0, 0, 0);

// Center on the electrode tip, then zoom in
scene.controls.center.set(0, 0, 12);
scene.controls.updateCenter();
scene.controls.mat.multiply(new modules.THREE.Matrix4().makeScale(5, 5, 5));
scene.controls.update();

// Listen for screenshot request from Python
scene.widget.model.on('msg:custom', (msg) => {
    if (msg.type === 'take_screenshot') {
        // scaling for higher res! the higher the better
        const scale = msg.scale || 1;
        const w = scene.renderer.domElement.clientWidth;
        const h = scene.renderer.domElement.clientHeight;
        const origRatio = scene.renderer.getPixelRatio();
        scene.renderer.setPixelRatio(origRatio * scale);
        scene.renderer.setSize(w, h, false);
        const img = scene.renderToImage();
        scene.renderer.setPixelRatio(origRatio);
        scene.renderer.setSize(w, h, false);
        scene.widget.send({type: 'screenshot', data: img.src,
                           file_name: msg.file_name});
    }
});

scene.animate();
"""


def on_screenshot(widget, content, buffers):
    """Trigger saving the screenshot upon message."""
    if content.get("type") == "screenshot":
        widget.on_msg(
            on_screenshot, remove=True
        )  # prevent handler accumulation on reruns
        file_name = content.get("file_name", "electrode_tip.png")
        img_data = base64.b64decode(content["data"].split(",")[1])
        with open(file_name, "wb") as f:
            f.write(img_data)
        print(f"Saved {file_name}")

In [ ]:
# euler_angles: [-90, 20, 0] rotates the electrode to appear vertical (tip at bottom)
# and adds a 20-degree Y rotation for a 3D perspective
scene = Draw(
    mesh,
    settings=draw_settings,
    euler_angles=[-90, -20, 0],
    js_code=js_code,
    width="1000px",
    height="1000px",
)

In [ ]:
filename = "electrode_tip_uncurved.png"
scene.widget.on_msg(on_screenshot)
scene.widget.send({"file_name": filename, "type": "take_screenshot"})
await asyncio.sleep(4)  # yield to event loop so the response can be processed
print("Saved to:", os.path.abspath(filename))

In [ ]:
scene2 = Draw(
    mesh.Curve(2),
    settings=draw_settings,
    euler_angles=[-90, -20, 0],
    js_code=js_code,
    width="1000px",
    height="1000px",
)

In [ ]:
filename = "electrode_tip_curved.png"
scene2.widget.on_msg(on_screenshot)
scene2.widget.send({"type": "take_screenshot", "file_name": filename})
await asyncio.sleep(4)  # yield to event loop so the response can be processed
print("Saved to:", os.path.abspath(filename))